# Measuring Feature-Level Forgetting

We train a small CNN on task A (CIFAR-10 classes 0–4), then on task B (classes 5–9), in task-incremental mode (we have two separate classifer heads for two tasks). We train a sparse autoencoder (SAE) on the model's penultimate-layer activations at the end of A and use it as a fixed "ruler" for features. For each important feature, we measure whether its information was **lost** (U), **moved** (M), or changed geometrically (fading, overlap, rotation). Thresholds come from a **null** run (A → A′) in which nothing should be forgotten. We repeat with several model seeds and SAE seeds to measure **reliability**, and validate everything on a **toy world** where the true features are known.

**How to run.**
1. **Software check first (recommended):** set `DATA = "synthetic"` and `SMOKE = True` (don't forget to set it back later).
2. `DATA = "cifar10"` (downloads CIFAR-10 to `~/data`, **it is mine path on server, change it to your Google Drive**).
3. Run all cells top to bottom. Models, activations and SAEs are cached in `OUT_DIR`, **save them to your Google Drive if you want to re-run code fast!).

Requirements: `torch`, `torchvision`, `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, all of that should already be in Google Colab

**It is a simplified version! The next step would be addressing that**: one layer (penultimate), one architecture (small CNN), ReLU-L1 SAE only (no TopK variant, no crosscoders), no residual SAE for newly learned B features, the global Procrustes rotation is fitted over all live features, and fate thresholds are taken from all live features of the null run (more stable than only the top 5%).

### The method

![How the measurement works](https://krutsylo.neocities.org/share/measurement_method.png)

**The question.** When a network learns task B after task A, its accuracy on A drops. Accuracy tells us that something broke, not what. But we need to figure out for each important internal **feature** of task A: was its information lost, did it just move, or did it survive? And can we trust the answer?

**1. Train three checkpoints.** From the model trained on A (**M_A**) we make two continuations. **M_B** learns task B: this is the real forgetting. **M_A′** keeps training on A with a small learning rate: nothing should be forgotten, so anything we measure there is noise. Task A's classifier head is never trained again, so every change in A's accuracy comes from the shared features.

**2. Fix a ruler on model A.** A sparse autoencoder (SAE) is trained once on M_A's activations and then frozen. Its encoder turns any input into **codes**: a few non-zero numbers saying which features are present. The codes are always computed from M_A, so "feature *i*" keeps the same meaning in every comparison.

**3. Measure each feature, on every transition** (A→A, A→A′, A→B):

| Question | How | Output |
|---|---|---|
| Can the information still be read? | Linear probes predict the feature's code from activations, scored by R² | **U** = R²(before) − R²(new probe): information **lost**. **M** = R²(new probe) − R²(old probe): information that is still there but **moved**, so the old readout misses it |
| Where is the feature written? | Regress each model's activations on the fixed codes; the coefficients are the feature vectors φ | Change in length (**fading** if shorter), crowding by other features (**overlap**), and **rotation** |
| What happened to it? | First rule that applies | **degraded** (U too high; sub-typed fading or overlap) → **misaligned** (M too high) → **amplified** → **retained** |

**4. Decide what can be trusted.** No threshold is chosen by hand.
* **A vs A** must give exactly zero. If not, the code has a bug.
* The **A vs A′ null** sets every threshold at its 95th percentile, and its noise must stay below 25% of the real signal.
* The whole measurement is repeated with **five SAE seeds**: a metric must give the same answer each time (ICC ≥ 0.7, match rate ≥ 0.5).
* In a **toy world** with known features, the metric must agree with the truth (Spearman ≥ 0.8).

A metric that passes the last two becomes a **primary** label. Otherwise it is **exploratory**: shown, but not used for claims. The notebook ends with eight pass/fail checks and an overall status.

*In the figure, the bars are real values for one important feature from a CIFAR-10 run: R² falls from 0.86 to 0.58 with a new probe (U ≈ 0.29) and to 0.13 with the old probe (M ≈ 0.45).*

### How this measurement differs from Putting a Face to Forgetting

Masip et al. (2026) introduced the feature-centric framework for forgetting: they interpret forgetting as changes in feature-vector norm, overlap/capacity, and downstream readout alignment, and demonstrated these ideas in a ViT using crosscoders. Their practical result, that fading is a dominant correlate of forgetting, was explicitly presented as a preliminary case study, and they note that SAE-based feature decompositions are approximate and sensitive to modeling choices.

We keep the same concept but turn it into a validated measurement protocol. Instead of learning a shared feature basis jointly across the before/after models, it anchors the feature definition on model A and measures every later checkpoint using the same fixed codes. It also adds functional tests that distinguish unrecoverable information loss from recoverable readout misalignment (if that will work, not confirmed so far), rather than relying only on geometric changes such as norm and capacity.

The main novelty is therefore measurement validation, not a new theory of forgetting: null transitions establish a noise floor, toy models provide known ground truth, repeated SAE runs quantify feature and metric reliability, and an artifact grid tests whether conclusions such as “fading dominates” survive changes in the feature-extraction method. The goal is to determine not only what forgetting looks like, but also whether the measurement itself can be trusted. It means: before treating a result like “this feature faded by 30%” as a real property of the neural network, we should establish that the number is not mainly caused by the measurement procedure itself.

For example, suppose the SAE/crosscoder reports that a feature's vector became much smaller after learning task B. There are at least two possibilities:
1. The network really changed the feature, which is a genuine fading.
2. The feature-extraction method changed or has a bias that makes vectors look smaller, which is apparent fading caused by the measurement.

The proposed protocol tries to distinguish these.


## Step 0 — Setup and configuration

Every number used later is set here, so nothing is hand-tuned after seeing results.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### [TRACKIO] Experiment tracking (added)

This notebook now sends its results to the team's **Trackio** dashboard (a free, self-hosted experiment tracker whose commands mirror Weights & Biases). **No experiment setting was changed**: same data, same 1 model seed, same 5 SAE seeds, same thresholds. Every added line is marked `[TRACKIO]` with a comment saying what it logs.

**How it works.** The cell after the config cell starts **one Trackio run** per full notebook execution, named like `cnn-sae-cifar10-ms1-ss5-20261025-1830`, with the whole `CFG` attached as its config. Each later step logs its results with a small helper, `tlog(...)`. The verdict cell writes a one-page report and closes the run.

**If the server is unreachable**, a warning is printed and the notebook runs exactly as before, just without logging. Trackio can never stop the experiment.

**Where to look:** open the server in a browser, then pick the project `forgetting-measurement`. Metric names are grouped by the prefix before the `/`:

| Prefix | What it holds | Logged in |
|---|---|---|
| `acc/` | test accuracies of M_A, M_A′, M_B and forgetting on A | Step 1 |
| `sae_ms0_ss0/` | SAE quality (L1, mean L0, FVU) and dead/live counts | Steps 3–4 |
| `demo_ms0_ss0/`, `null_ms0_ss0/`, `fate_count_ms0_ss0_*/` | seed-0 demo, null thresholds, noise gate, fate counts | Steps 5–10 demo |
| `seeds_ms0/` | per-SAE-seed checks, drawn as lines across SAE seeds (x-axis = SAE seed) | main loop |
| `fate_share_AB/`, `fate_share_null/`, `hist_real/`, `hist_null/`, `figures/` | pooled fate shares, histograms, the U/M/log-norm figure | after main loop |
| `reliability/`, `icc/` | match rates, Fleiss' kappa, ICC per metric | Step 11 |
| `toy/`, `toy_mixed/`, `toy_fp/` | toy-world validation | Step 12 |
| `primary/`, `check/`, `result/`, `final_fate_share/` | reliability rule, the 8 checks (1 = pass), headline results | rule + verdict |
| `tables/` | every summary table (run checks, fate shares, ICC, toy, rule, diagnostic, verdict) | throughout |
| `report/verdict` | one-page Markdown summary (dashboard **Reports** page) | verdict |

**Steps (the x-axis).** One-off results are logged at step 0. Per-seed results use the seed number as the step, so they draw as a line across seeds.

In [ ]:
# ===================== [TRACKIO] ADDED: install the Trackio client =====================
# Installs the Trackio Python package on this Colab machine only (it does not touch your Drive).
# Pinned to 0.43.0, the version this notebook's Trackio code was tested against, so every teammate
# runs the same client. If the team server reports a version mismatch, change the number to the
# server's version. If pip prints "restart the runtime", do Runtime > Restart session, then run
# all cells from the top again.
%pip install -q trackio==0.43.0
import importlib.metadata
print("trackio client version:", importlib.metadata.version("trackio"))

In [ ]:
import os, json, time, math, copy, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from scipy.optimize import linear_sum_assignment
from scipy.linalg import orthogonal_procrustes
from scipy.special import logsumexp
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

DATA = "cifar10"     # "cifar10" for the real run, "synthetic" for a fast software test
QUICK = False      # True = small, fast test on CIFAR10, slower than synthetic
SMOKE = False     # True = tiny settings that only check the code runs end to end

CFG = dict(
    DATA=DATA, QUICK=QUICK, SMOKE=SMOKE,
    SYN_N_TRAIN=1000, SYN_N_TEST=300,     # synthetic images per class
    DATA_ROOT="/content/drive/MyDrive/forgetting_project_V1/data", # CHANGE THAT!
    OUT_DIR=f"/content/drive/MyDrive/forgetting_project_V1/ff_results_{DATA}{'_smoke' if SMOKE else ('_quick' if QUICK else '')}",
    DEVICE="cuda" if torch.cuda.is_available() else "cpu",
    NUM_WORKERS=2,
    # models
    N_MODEL_SEEDS=8, WIDTH=64,         # [vwaugh] 8 model seeds (0-7); was 1
    EPOCHS_A=15, EPOCHS_B=15, LR=1e-3, WD=0.05, BATCH=128,
    HEAD_PROBE_N=5000,                 # images used for the linear-probe init of B's head (H-LP)
    # SAE (the anchored dictionary)
    N_SAE_SEEDS=5, SAE_MULT=2, TARGET_L0=16,
    SAE_EPOCHS=40, SAE_CALIB_ROUNDS=10, SAE_CALIB_EPOCHS=8, SAE_MIN_STEPS=5000,
    # data used by the measurement
    N_PROBE_TRAIN=5000, N_HELDOUT_PER_TASK=2500,
    # feature selection
    MIN_ACTIVATIONS=100, TOP_FRAC=0.2, MIN_FEATURES=5, N_NEIGHBOURS=10,
    # thresholds, gates and rules
    NULL_PCT=95, NOISE_GATE=0.25,
    RULE_ICC=0.7, RULE_MATCH=0.5, RULE_MATCH_COS=0.7, RULE_TOY_SPEARMAN=0.8,
    # toy world
    TOY_SEEDS=5,
)
if QUICK:
    CFG.update(N_MODEL_SEEDS=2, N_SAE_SEEDS=3, WIDTH=32, EPOCHS_A=4, EPOCHS_B=4,
               SAE_EPOCHS=15, SAE_CALIB_ROUNDS=8, SAE_CALIB_EPOCHS=4,
               HEAD_PROBE_N=2000, MIN_ACTIVATIONS=100, TOY_SEEDS=2, NUM_WORKERS=0)
if SMOKE:
    CFG.update(N_MODEL_SEEDS=2, N_SAE_SEEDS=2, WIDTH=8, EPOCHS_A=2, EPOCHS_B=2, BATCH=64,
               SYN_N_TRAIN=300, SYN_N_TEST=120, N_PROBE_TRAIN=600, N_HELDOUT_PER_TASK=400,
               SAE_EPOCHS=10, SAE_CALIB_ROUNDS=8, SAE_CALIB_EPOCHS=3, SAE_MIN_STEPS=800, TARGET_L0=8,
               HEAD_PROBE_N=500, MIN_ACTIVATIONS=40, TOY_SEEDS=1, NUM_WORKERS=0)
if CFG["DEVICE"] == "cpu":
    CFG["NUM_WORKERS"] = 0
OUT = CFG["OUT_DIR"]
os.makedirs(OUT, exist_ok=True)
DEV = CFG["DEVICE"]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

print(json.dumps(CFG, indent=1))

In [ ]:
# ===================== [TRACKIO] ADDED: connect to the team's Trackio server and start ONE run =====================
# 1. Points Trackio at the team's self-hosted dashboard (TRACKIO_SERVER_URL).
# 2. Authenticates with the server's WRITE TOKEN, a password-like key that lets this notebook add data.
#    It is read from a Colab secret named TRACKIO_WRITE_TOKEN if one exists (🔑 icon in Colab's left
#    sidebar; safer, because the key then is not stored inside the shared notebook). Otherwise it
#    falls back to the key pasted below.
# 3. Starts ONE Trackio run for this whole notebook execution. The name holds the data mode, seed counts
#    and a timestamp, so runs never collide. `group` keeps smoke / quick / real (cifar10) runs apart.
#    CFG (minus file paths) is stored as the run's config, so every number on the dashboard is tied to
#    the exact settings that produced it.
# 4. Defines the helpers every later [TRACKIO] line uses:
#      tlog(dict, step=0) -> sends numbers / tables / figures to the server
#      ttable(df)         -> pandas DataFrame -> Trackio table (rounds numbers, fills blanks with "n/a")
#      tfig(fig)          -> matplotlib figure -> Trackio HTML figure
#      thist(values)      -> array of numbers -> native Trackio histogram
#      tmd(text)          -> Markdown text -> Trackio report
# Nothing here changes any experiment setting. If Trackio fails to start, USE_TRACKIO becomes False,
# a warning prints, and every later [TRACKIO] line quietly does nothing.
import re

TRACKIO_SERVER_URL = "http://194.53.86.83:7860"   # team's self-hosted Trackio dashboard
TRACKIO_PROJECT    = "forgetting-measurement_vwaugh"  # [vwaugh] own project, NOT forgetting-measurement_0
WHO                = "vwaugh"                  # <- change to your name so runs show who made them

_FALLBACK_WRITE_TOKEN = "PASTE_WRITE_TOKEN_HERE"   # used only if no Colab secret is set (token is in Slack; never commit it)
try:
    from google.colab import userdata
    _tok = userdata.get("TRACKIO_WRITE_TOKEN")
except Exception:                      # no secret set, or notebook access to it not granted
    _tok = None
os.environ["TRACKIO_WRITE_TOKEN"] = _tok or _FALLBACK_WRITE_TOKEN

RUN_GROUP = "smoke" if CFG["SMOKE"] else ("quick" if CFG["QUICK"] else CFG["DATA"])
RUN_NAME  = f"{WHO}-cnn-sae-{RUN_GROUP}-ms{CFG['N_MODEL_SEEDS']}-ss{CFG['N_SAE_SEEDS']}-{time.strftime('%Y%m%d-%H%M')}"
TRACK_CFG = {k: v for k, v in CFG.items() if k not in ("DATA_ROOT", "OUT_DIR", "DEVICE")}
TRACK_CFG.update(tool="sae_relu_l1", layer="penultimate", arch="small_cnn", who=WHO)   # labels for filtering runs later

USE_TRACKIO = True
try:
    import trackio
    try:
        trackio.finish()               # closes a run left open by an earlier, crashed execution of this cell
    except Exception:
        pass
    trackio.init(project=TRACKIO_PROJECT, name=RUN_NAME, group=RUN_GROUP, config=TRACK_CFG,
                 server_url=TRACKIO_SERVER_URL, embed=False)   # embed=False: don't draw the dashboard inside this cell
    print(f"Trackio run started: {RUN_NAME}  (group={RUN_GROUP})  ->  {TRACKIO_SERVER_URL}")
except Exception as e:
    USE_TRACKIO = False
    print("WARNING: Trackio could not start; the notebook continues WITHOUT tracking:", repr(e))


def _clean_value(v):
    # numbers -> plain float (True/False -> 1.0/0.0); NaN/inf -> None (skipped); Trackio objects pass through
    if isinstance(v, (bool, np.bool_)):
        return float(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        v = float(v)
        return v if math.isfinite(v) else None
    return v


def tlog(metrics, step=0):
    # [TRACKIO] Log a dict of results. One-off results use step=0; per-seed results pass step=<seed>.
    # Metric names are made dashboard-safe: any character other than letters, digits, _ / . - becomes "_"
    # (e.g. "degraded:fading" -> "degraded_fading"). Errors are printed and ignored, never raised.
    if not USE_TRACKIO:
        return
    clean = {}
    for k, v in metrics.items():
        v = _clean_value(v)
        if v is not None:
            clean[re.sub(r"[^A-Za-z0-9_/.\-]", "_", str(k))] = v
    if not clean:
        return
    try:
        trackio.log(clean, step=int(step))
    except Exception as e:
        print("WARNING: trackio.log failed (ignored):", repr(e))


def ttable(df, nd=4, keep_index=False):
    # [TRACKIO] DataFrame -> Trackio table. keep_index=True keeps row labels (e.g. fate names) as a column.
    if not USE_TRACKIO:
        return None
    try:
        d = (df.reset_index() if keep_index else df.reset_index(drop=True)).copy()
        num = d.select_dtypes(include="number").columns
        d[num] = d[num].round(nd)
        d = d.astype(object).where(d.notna(), "n/a")
        d.columns = [str(c) for c in d.columns]
        return trackio.Table(dataframe=d)
    except Exception as e:
        print("WARNING: ttable failed (ignored):", repr(e)); return None


def tfig(fig):
    # [TRACKIO] matplotlib figure -> Trackio HTML figure (shows on the dashboard as an image).
    if not USE_TRACKIO:
        return None
    try:
        return trackio.Html(fig)
    except Exception as e:
        print("WARNING: tfig failed (ignored):", repr(e)); return None


def thist(values):
    # [TRACKIO] array of numbers -> native Trackio histogram (NaN/inf dropped).
    if not USE_TRACKIO:
        return None
    try:
        x = np.asarray(values, dtype=float)
        x = x[np.isfinite(x)]
        return trackio.Histogram(x) if len(x) else None
    except Exception as e:
        print("WARNING: thist failed (ignored):", repr(e)); return None


def tmd(text):
    # [TRACKIO] Markdown text -> Trackio report (appears on the dashboard's Reports page).
    if not USE_TRACKIO:
        return None
    try:
        return trackio.Markdown(text)
    except Exception as e:
        print("WARNING: tmd failed (ignored):", repr(e)); return None

## Step 1 — Data, model, and training of the three checkpoints

For each model seed we produce three checkpoints:

| Checkpoint | How it is made | Role |
|---|---|---|
| **M_A** | train on task A | the "before" model |
| **M_A′** | continue training M_A on A (new data order, 0.1× learning rate, as many steps as a B task) | the **null**: nothing should be forgotten, so whatever we measure here is noise |
| **M_B** | continue training M_A on B (new head, initialised by a linear probe = protocol H-LP) | the "after" model |

Task A's head is never trained during B, so any change in A's accuracy comes from the shared backbone, which is exactly where features live.

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

class TaskView(torch.utils.data.Dataset):
    """The images of one task, with labels remapped to 0..4."""
    def __init__(self, base, idx, offset):
        self.base, self.idx, self.offset = base, np.asarray(idx), offset
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        x, y = self.base[int(self.idx[k])]
        return x, int(y) - self.offset

def make_synthetic(seed=1234, n_train=1000, n_test=300):
    """Image-like data with shared parts between classes (so tasks interfere). Smoke tests only."""
    g = np.random.RandomState(seed)
    n_parts = 12
    raw = torch.tensor(g.randn(n_parts, 3, 8, 8).astype(np.float32))
    parts = Fnn.interpolate(raw, size=32, mode="bilinear", align_corners=False).numpy()
    class_parts = [g.choice(n_parts, 3, replace=False) for _ in range(10)]
    def gen(n):
        X = np.zeros((10 * n, 3, 32, 32), np.float32)
        Y = np.repeat(np.arange(10), n)
        for c in range(10):
            amps = g.uniform(0.3, 1.5, size=(n, 3, 1, 1, 1)).astype(np.float32)
            imgs = (amps * parts[class_parts[c]][None]).sum(1)
            for k in range(n):
                imgs[k] = np.roll(imgs[k], shift=(g.randint(-4, 5), g.randint(-4, 5)), axis=(1, 2))
            X[c * n:(c + 1) * n] = imgs + 0.8 * g.randn(n, 3, 32, 32)
        return torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(Y))
    return gen(n_train), gen(n_test)

def get_data():
    if CFG["DATA"] == "cifar10":
        import torchvision, torchvision.transforms as T
        norm = T.Normalize(CIFAR_MEAN, CIFAR_STD)
        aug = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), norm])
        plain = T.Compose([T.ToTensor(), norm])
        tr_aug = torchvision.datasets.CIFAR10(CFG["DATA_ROOT"], train=True, download=True, transform=aug)
        tr_plain = torchvision.datasets.CIFAR10(CFG["DATA_ROOT"], train=True, download=True, transform=plain)
        te_plain = torchvision.datasets.CIFAR10(CFG["DATA_ROOT"], train=False, download=True, transform=plain)
        ytr, yte = np.array(tr_plain.targets), np.array(te_plain.targets)
    else:
        tr, te = make_synthetic(n_train=CFG["SYN_N_TRAIN"], n_test=CFG["SYN_N_TEST"])
        tr_aug = tr_plain = tr
        te_plain = te
        ytr, yte = tr.tensors[1].numpy(), te.tensors[1].numpy()
    D = {}
    for t, (lo, hi) in {"A": (0, 5), "B": (5, 10)}.items():
        itr = np.where((ytr >= lo) & (ytr < hi))[0]
        ite = np.where((yte >= lo) & (yte < hi))[0]
        D[t] = dict(train_aug=TaskView(tr_aug, itr, lo), train_plain=TaskView(tr_plain, itr, lo),
                    test=TaskView(te_plain, ite, lo), n_train=len(itr))
    return D

DATA_D = get_data()

# Fixed index sets (positions inside each task's training set), drawn once and reused everywhere
_rng = np.random.RandomState(0)
_pA = _rng.permutation(DATA_D["A"]["n_train"]); _pB = _rng.permutation(DATA_D["B"]["n_train"])
_nP = min(CFG["N_PROBE_TRAIN"], len(_pA) // 2)
_nH = min(CFG["N_HELDOUT_PER_TASK"], len(_pA) - _nP, len(_pB))
IDX = dict(A_probe=np.sort(_pA[:_nP]), A_held=np.sort(_pA[_nP:_nP + _nH]), B_held=np.sort(_pB[:_nH]))
print({k: len(v) for k, v in IDX.items()}, "| A train:", DATA_D["A"]["n_train"], "| A test:", len(DATA_D["A"]["test"]))

In [ ]:
def conv_block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class MultiHeadCNN(nn.Module):
    """Shared backbone + one linear head per task. The backbone output is the 'penultimate layer'."""
    def __init__(self, width=64):
        super().__init__()
        self.backbone = nn.Sequential(conv_block(3, width), conv_block(width, 2 * width),
                                      conv_block(2 * width, 4 * width),
                                      nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.feat_dim = 4 * width
        self.heads = nn.ModuleDict()
    def add_head(self, task, n_classes=5):
        self.heads[task] = nn.Linear(self.feat_dim, n_classes).to(next(self.backbone.parameters()).device)
    def features(self, x):
        return self.backbone(x)
    def forward(self, x, task):
        return self.heads[task](self.features(x))

def make_loader(ds, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(ds, batch_size=CFG["BATCH"], shuffle=shuffle,
                                       generator=g if shuffle else None,
                                       num_workers=CFG["NUM_WORKERS"], drop_last=shuffle)

def train_task(model, task, ds, steps, lr, data_seed):
    """Train the backbone + the current task's head only. Older heads are untouched."""
    params = list(model.backbone.parameters()) + list(model.heads[task].parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(steps, 1))
    model.train(); step, epoch = 0, 0
    while step < steps:
        for x, y in make_loader(ds, True, seed=data_seed + epoch):
            x, y = x.to(DEV), y.to(DEV)
            loss = Fnn.cross_entropy(model(x, task), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            step += 1
            if step >= steps:
                break
        epoch += 1
    model.eval()
    return model

@torch.no_grad()
def get_acts(model, ds, idx=None, labels=False):
    """Penultimate-layer activations (and optionally labels) for a dataset or a subset of it."""
    if idx is not None:
        ds = torch.utils.data.Subset(ds, [int(i) for i in idx])
    model.eval(); A, Y = [], []
    for x, y in make_loader(ds, False):
        A.append(model.features(x.to(DEV)).float().cpu()); Y.append(torch.as_tensor(y))
    A = torch.cat(A).numpy().astype(np.float64)
    return (A, torch.cat(Y).numpy()) if labels else A

@torch.no_grad()
def accuracy(model, task, ds):
    model.eval(); c = n = 0
    for x, y in make_loader(ds, False):
        c += (model(x.to(DEV), task).argmax(1).cpu() == y).sum().item(); n += len(y)
    return c / n

def init_head_linear_probe(model, task, ds_plain, seed):
    """Protocol H-LP: fit B's head as a logistic regression on frozen features before joint training."""
    idx = np.random.RandomState(seed).choice(len(ds_plain), min(CFG["HEAD_PROBE_N"], len(ds_plain)), replace=False)
    X, y = get_acts(model, ds_plain, idx, labels=True)
    clf = LogisticRegression(C=1.0, max_iter=2000).fit(X, y)
    with torch.no_grad():
        model.heads[task].weight.copy_(torch.tensor(clf.coef_, dtype=torch.float32))
        model.heads[task].bias.copy_(torch.tensor(clf.intercept_, dtype=torch.float32))

def train_checkpoints(ms):
    path = f"{OUT}/models_seed{ms}.pt"
    def fresh(with_b=False):
        m = MultiHeadCNN(CFG["WIDTH"]).to(DEV); m.add_head("A")
        if with_b: m.add_head("B")
        return m
    if os.path.exists(path):
        ck = torch.load(path, map_location=DEV)
        M = {"A": fresh(), "Aprime": fresh(), "B": fresh(True)}
        for k in M: M[k].load_state_dict(ck[k]); M[k].eval()
        return M, ck["accs"]
    seed_all(ms)
    t0 = time.time()
    m = fresh()
    steps_A = CFG["EPOCHS_A"] * (DATA_D["A"]["n_train"] // CFG["BATCH"])
    steps_B = CFG["EPOCHS_B"] * (DATA_D["B"]["n_train"] // CFG["BATCH"])
    train_task(m, "A", DATA_D["A"]["train_aug"], steps_A, CFG["LR"], data_seed=1000 * ms)
    M_A = copy.deepcopy(m)
    M_Ap = copy.deepcopy(M_A)   # null: same number of steps as B, new data order, 0.1x LR
    train_task(M_Ap, "A", DATA_D["A"]["train_aug"], steps_B, 0.1 * CFG["LR"], data_seed=1000 * ms + 500)
    M_B = copy.deepcopy(M_A); M_B.add_head("B")
    init_head_linear_probe(M_B, "B", DATA_D["B"]["train_plain"], seed=ms)
    train_task(M_B, "B", DATA_D["B"]["train_aug"], steps_B, CFG["LR"], data_seed=1000 * ms + 900)
    accs = dict(acc_A_after_A=accuracy(M_A, "A", DATA_D["A"]["test"]),
                acc_A_after_Aprime=accuracy(M_Ap, "A", DATA_D["A"]["test"]),
                acc_A_after_B=accuracy(M_B, "A", DATA_D["A"]["test"]),
                acc_B_after_B=accuracy(M_B, "B", DATA_D["B"]["test"]))
    torch.save({"A": M_A.state_dict(), "Aprime": M_Ap.state_dict(), "B": M_B.state_dict(), "accs": accs}, path)
    print(f"  model seed {ms} trained in {time.time() - t0:.0f}s")
    return {"A": M_A, "Aprime": M_Ap, "B": M_B}, accs

MODELS, ACCS = {}, {}
for ms in range(CFG["N_MODEL_SEEDS"]):
    MODELS[ms], ACCS[ms] = train_checkpoints(ms)
acc_df = pd.DataFrame(ACCS).T
acc_df.index.name = "model_seed"
acc_df["forgetting_on_A"] = acc_df["acc_A_after_A"] - acc_df["acc_A_after_B"]

# [TRACKIO] ADDED: log the test accuracies of the three checkpoints and the forgetting on A
# (one point per model seed; step = model seed).
for _ms, _row in acc_df.iterrows():
    tlog({f"acc/{k}": v for k, v in _row.items()}, step=int(_ms))
acc_df

**What to look for:** accuracy on A should drop clearly after B (that is the forgetting we want to explain), while A after A′ should stay close to A after A (the null should forget almost nothing).

## Step 2 — Collect activations

For every checkpoint we record penultimate-layer activations on: all of A's training images (for the SAE and the probes), A's test images (to evaluate probes and importance), and a fixed **held-out** set mixing A and B images 1:1 (for the anchored regression). The same images are used for every checkpoint.

In [ ]:
def checkpoint_acts(model):
    A_train = get_acts(model, DATA_D["A"]["train_plain"])
    A_test, y_test = get_acts(model, DATA_D["A"]["test"], labels=True)
    held = np.concatenate([A_train[IDX["A_held"]], get_acts(model, DATA_D["B"]["train_plain"], IDX["B_held"])])
    return dict(A_train=A_train, A_probe=A_train[IDX["A_probe"]], A_test=A_test, y_test=y_test, held=held)

def load_acts(ms):
    path = f"{OUT}/acts_seed{ms}.npz"
    if os.path.exists(path):
        z = np.load(path)
        return {ck: {k.split("__")[1]: z[k] for k in z.files if k.startswith(ck + "__")} for ck in ["A", "Aprime", "B"]}
    acts = {ck: checkpoint_acts(MODELS[ms][ck]) for ck in ["A", "Aprime", "B"]}
    np.savez(path, **{f"{ck}__{k}": v for ck, d in acts.items() for k, v in d.items()})
    return acts

ACTS = {ms: load_acts(ms) for ms in range(CFG["N_MODEL_SEEDS"])}
HEAD_A = {ms: (MODELS[ms]["A"].heads["A"].weight.detach().cpu().numpy().astype(np.float64),
               MODELS[ms]["A"].heads["A"].bias.detach().cpu().numpy().astype(np.float64))
          for ms in range(CFG["N_MODEL_SEEDS"])}
print({k: v.shape for k, v in ACTS[0]["A"].items()})

## Step 3 — Train the anchored SAE on model A

The SAE is trained **only** on model A's activations of A's training images, then frozen forever. Activations are rescaled by one constant so their average squared length equals their dimension; this makes the L1 strength comparable across runs. The L1 strength is calibrated automatically so that on average about `TARGET_L0` features are active per image.

In [ ]:
class SAE(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.enc = nn.Linear(d, m)
        self.dec = nn.Linear(m, d)
        with torch.no_grad():
            w = torch.randn(d, m); w /= w.norm(dim=0, keepdim=True)
            self.dec.weight.copy_(w); self.enc.weight.copy_(w.T); self.enc.bias.zero_()
    def encode(self, x):
        return torch.relu(self.enc(x - self.dec.bias))
    def forward(self, x):
        f = self.encode(x)
        return self.dec(f), f

def fit_sae(X, l1, epochs, seed, min_steps=None):
    seed_all(seed)
    Xt = torch.tensor(X, dtype=torch.float32, device=DEV)
    sae = SAE(Xt.shape[1], CFG_SAE_MULT * Xt.shape[1]).to(DEV)
    with torch.no_grad():
        sae.dec.bias.copy_(Xt.mean(0))
    opt = torch.optim.Adam(sae.parameters(), lr=1e-3)
    g = torch.Generator(); g.manual_seed(seed)
    n_batches = math.ceil(len(Xt) / 256)
    epochs = max(epochs, math.ceil((min_steps or CFG["SAE_MIN_STEPS"]) / n_batches))   # never train for too few steps
    for _ in range(epochs):
        for xb in Xt[torch.randperm(len(Xt), generator=g).to(DEV)].split(256):
            recon, f = sae(xb)
            loss = ((recon - xb) ** 2).sum(-1).mean() + l1 * (f * sae.dec.weight.norm(dim=0)).sum(-1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return sae.cpu().eval()

@torch.no_grad()
def sae_stats(sae, X):
    Xt = torch.tensor(X, dtype=torch.float32)
    recon, f = sae(Xt)
    fvu = ((recon - Xt) ** 2).sum() / ((Xt - Xt.mean(0)) ** 2).sum()
    return float((f > 0).float().sum(1).mean()), float(fvu)

def calibrate_l1(X, target, rounds, epochs, seed, l1=1.0, min_steps=None):
    """Search (bracketing, then bisection in log space) for the L1 strength giving the target mean L0."""
    lo = hi = None
    for r in range(rounds):
        L0, fvu = sae_stats(fit_sae(X, l1, epochs, seed, min_steps), X)
        if abs(L0 - target) / target < 0.15:
            break
        if L0 > target:          # too many active features -> penalty too weak
            lo = l1
            l1 = l1 * 4 if hi is None else math.sqrt(lo * hi)
        else:                    # too few -> penalty too strong
            hi = l1
            l1 = l1 / 4 if lo is None else math.sqrt(lo * hi)
    return l1

@torch.no_grad()
def codes(sae, X, scale):
    """Fixed feature codes f(x) = E_A(a_A(x)). Always applied to MODEL A's activations."""
    return sae.encode(torch.tensor(X / scale, dtype=torch.float32)).numpy().astype(np.float64)

CFG_SAE_MULT = CFG["SAE_MULT"]

def get_sae(ms, ss, X_A_train, target_l0=None, tag="cnn", epochs=None, min_steps=None):
    target_l0 = target_l0 or CFG["TARGET_L0"]
    epochs = epochs or CFG["SAE_EPOCHS"]
    path = f"{OUT}/sae_{tag}_m{ms}_s{ss}.pt"
    scale = float(np.sqrt((X_A_train ** 2).sum(1).mean() / X_A_train.shape[1]))
    Xs = X_A_train / scale
    if os.path.exists(path):
        ck = torch.load(path)
        sae = SAE(Xs.shape[1], CFG_SAE_MULT * Xs.shape[1]); sae.load_state_dict(ck["state"]); sae.eval()
        return sae, scale, ck["l1"]
    l1_path = f"{OUT}/sae_{tag}_m{ms}_l1.json"
    if os.path.exists(l1_path):
        l1 = json.load(open(l1_path))["l1"]
    else:   # calibrate once per model seed (with SAE seed 0) and reuse for all SAE seeds
        l1 = calibrate_l1(Xs, target_l0, CFG["SAE_CALIB_ROUNDS"], CFG["SAE_CALIB_EPOCHS"], seed=10_000 + ms, min_steps=min_steps)
        json.dump({"l1": l1}, open(l1_path, "w"))
    sae = fit_sae(Xs, l1, epochs, seed=100 * ms + ss, min_steps=min_steps)
    torch.save({"state": sae.state_dict(), "l1": l1}, path)
    return sae, scale, l1

t0 = time.time()
sae0, scale0, l1_0 = get_sae(0, 0, ACTS[0]["A"]["A_train"])
L0, fvu = sae_stats(sae0, ACTS[0]["A"]["A_train"] / scale0)
print(f"SAE (model seed 0, SAE seed 0): l1={l1_0:.3g}  mean L0={L0:.1f}  "
      f"fraction of variance unexplained={fvu:.3f}  ({time.time() - t0:.0f}s)")

# [TRACKIO] ADDED: log SAE quality for model seed 0 / SAE seed 0: the calibrated L1 penalty strength, the average
# number of features active per image (mean L0, aimed at TARGET_L0) and the fraction of variance unexplained (FVU).
tlog({"sae_ms0_ss0/l1": l1_0, "sae_ms0_ss0/mean_L0": L0, "sae_ms0_ss0/target_L0": CFG["TARGET_L0"],
      "sae_ms0_ss0/fvu": fvu})

**What to look for:** mean L0 near the target and a fraction of variance unexplained well below 1 (e.g. under 0.3). If FVU is high, the SAE is too sparse or undertrained: increase `SAE_EPOCHS` or `TARGET_L0`.

## Step 4 — Compute the fixed codes

Codes are always computed from **model A's** activations, whichever checkpoint we study later. That is what keeps each feature's identity fixed.

In [ ]:
F_demo = dict(train=codes(sae0, ACTS[0]["A"]["A_train"], scale0),
              test=codes(sae0, ACTS[0]["A"]["A_test"], scale0),
              held=codes(sae0, ACTS[0]["A"]["held"], scale0))
counts = (F_demo["test"] > 0).sum(0)
print(f"{F_demo['train'].shape[1]} latents | dead on A test: {(counts == 0).sum()} | "
      f"with >= {CFG['MIN_ACTIVATIONS']} activations on A test: {(counts >= CFG['MIN_ACTIVATIONS']).sum()}")

# [TRACKIO] ADDED: log how many SAE features are dead (never fire on A's test images) vs live
# (fire at least MIN_ACTIVATIONS times). Only live features are analysed later.
tlog({"sae_ms0_ss0/n_latents": F_demo["train"].shape[1], "sae_ms0_ss0/n_dead": (counts == 0).sum(),
      "sae_ms0_ss0/n_live": (counts >= CFG["MIN_ACTIVATIONS"]).sum()})

## Steps 5–8 — The measurement functions

The next cell defines all the measurement logic as small, plain-NumPy functions. They are reused unchanged for the CNN, for the null runs and for the toy world, so every result in this notebook comes from the same code.

* **Step 5 — Feature selection.** A latent is *live* if it fires at least `MIN_ACTIVATIONS` times on A's test images. Its **importance** is how much A's cross-entropy rises when we remove its contribution f_i(x)·φ_i from the activation (everything else, including the SAE's reconstruction error, stays in place). We analyse the top 5% of live latents.
* **Step 6 — Anchored regression.** At any checkpoint, the feature vectors Φ are the coefficients of a ridge regression *activations ≈ Φ·codes + b*, fitted on the held-out set. Same codes, same inputs and same ridge for every checkpoint.
* **Step 7 — Readability.** Ridge probes predict each feature's code from activations: R²_A (fit and test on A), R²_frozen (A's probe applied to the new model), R²_refit (new probe on the new model). Then **U = R²_A − R²_refit** (information lost) and **M = R²_refit − R²_frozen** (information moved). **U_own** repeats U with partial R², i.e. only the information beyond the feature's 10 most correlated neighbours.
* **Step 8 — Geometry.** On a common scale (each vector multiplied by its feature's RMS code): log-norm ratio (fading < 0 < amplification), log-capacity ratio (overlap makes it negative), rotation split into a global part (one Procrustes rotation for all features) and a local part, and the relative misalignment with A's head.

In [ ]:
# ---------- small linear-algebra helpers (closed form, deterministic) ----------
def ridge_fit(X, Y, alpha):
    mx, my = X.mean(0), Y.mean(0)
    Xc = X - mx
    B = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ (Y - my))
    return B, my - mx @ B

def r2_cols(Y, P):
    ss_res = ((Y - P) ** 2).sum(0)
    ss_tot = ((Y - Y.mean(0)) ** 2).sum(0)
    return 1.0 - ss_res / np.maximum(ss_tot, 1e-12)

def choose_alpha(X, Y, n_folds=5, seed=0):
    """Probe strength by 5-fold cross-validation, chosen ONCE on model A and then frozen."""
    base = np.trace(np.cov(X.T)) * len(X) / X.shape[1]
    grid = base * np.logspace(-5, 1, 13)
    folds = np.array_split(np.random.RandomState(seed).permutation(len(X)), n_folds)
    scores = []
    for a in grid:
        s = []
        for k in range(n_folds):
            te = folds[k]; tr = np.concatenate([folds[j] for j in range(n_folds) if j != k])
            B, c = ridge_fit(X[tr], Y[tr], a * (n_folds - 1) / n_folds)
            s.append(np.nanmean(r2_cols(Y[te], X[te] @ B + c)))
        scores.append(np.mean(s))
    return float(grid[int(np.argmax(scores))])

# ---------- Step 5: live features and importance ----------
def select_features(F_test, acts_test, y_test, Phi_A, W, b):
    counts = (F_test > 0).sum(0)
    live = np.where(counts >= CFG["MIN_ACTIVATIONS"])[0]
    logits0 = acts_test @ W.T + b
    base_ce = np.mean(logsumexp(logits0, 1) - logits0[np.arange(len(y_test)), y_test])
    WPhi = Phi_A @ W.T                         # effect of each feature on the logits
    imp = np.full(F_test.shape[1], np.nan)
    for chunk in np.array_split(live, max(1, len(live) // 32)):
        L = logits0[None] - F_test[:, chunk].T[:, :, None] * WPhi[chunk][:, None, :]
        ce = logsumexp(L, 2) - L[:, np.arange(len(y_test)), y_test]
        imp[chunk] = ce.mean(1) - base_ce
    k = min(len(live), max(CFG["MIN_FEATURES"], int(math.ceil(CFG["TOP_FRAC"] * len(live)))))
    top = live[np.argsort(-imp[live])[:k]]
    return live, top, imp

# ---------- Step 6: anchored regression ----------
def anchored_vectors(F_held, acts_held):
    eps = 1e-3 * np.mean(F_held.var(0)) * len(F_held)
    Phi, b = ridge_fit(F_held, acts_held, eps)
    return Phi, b                               # Phi[i] = feature vector of feature i

# ---------- Step 7: readability, U, M, U_own ----------
def readability(XA_tr, XA_te, XX_tr, XX_te, Y_tr, Y_te, alpha):
    BA, cA = ridge_fit(XA_tr, Y_tr, alpha)
    BX, cX = ridge_fit(XX_tr, Y_tr, alpha)
    R2_A = r2_cols(Y_te, XA_te @ BA + cA)
    R2_frozen = r2_cols(Y_te, XX_te @ BA + cA)
    R2_refit = r2_cols(Y_te, XX_te @ BX + cX)
    return dict(R2_A=R2_A, R2_frozen=R2_frozen, R2_refit=R2_refit,
                U=R2_A - R2_refit, M=R2_refit - R2_frozen)

def neighbours(F_tr_live, k):
    C = np.corrcoef(F_tr_live.T); np.fill_diagonal(C, -np.inf); C = np.nan_to_num(C, nan=-np.inf)
    return np.argsort(-C, 1)[:, :k]

def partial_r2(X_tr, X_te, Fl_tr, Fl_te, nbrs, alpha):
    """Partial R^2 of each feature: share of its variance explained by activations BEYOND its neighbours' codes.
    Ridge on the activations, unpenalised neighbours + intercept (Frisch-Waugh-Lovell, exact)."""
    XtX = X_tr.T @ X_tr; I = np.eye(X_tr.shape[1]); out = np.zeros(Fl_tr.shape[1])
    for j in range(Fl_tr.shape[1]):
        Z = np.hstack([Fl_tr[:, nbrs[j]], np.ones((len(X_tr), 1))])
        Zte = np.hstack([Fl_te[:, nbrs[j]], np.ones((len(X_te), 1))])
        y, yte = Fl_tr[:, j], Fl_te[:, j]
        ZtZi = np.linalg.pinv(Z.T @ Z)
        g_nb = ZtZi @ (Z.T @ y)
        r_nb = r2_cols(yte[:, None], (Zte @ g_nb)[:, None])[0]
        XtZ = X_tr.T @ Z
        beta = np.linalg.solve(XtX - XtZ @ ZtZi @ XtZ.T + alpha * I, X_tr.T @ y - XtZ @ g_nb)
        gam = ZtZi @ (Z.T @ (y - X_tr @ beta))
        r_full = r2_cols(yte[:, None], (X_te @ beta + Zte @ gam)[:, None])[0]
        out[j] = (r_full - r_nb) / max(1.0 - r_nb, 1e-6)
    return out

# ---------- Step 8: geometry ----------
def capacity(P):
    G = P @ P.T
    return np.diag(G) ** 2 / np.maximum((G ** 2).sum(1), 1e-30)

def angle(U, V):
    cos = (U * V).sum(1) / np.maximum(np.linalg.norm(U, axis=1) * np.linalg.norm(V, axis=1), 1e-30)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def geometry(Phi_A, Phi_X, rms, live, W=None):
    PA, PX = Phi_A * rms[:, None], Phi_X * rms[:, None]      # common scale
    nA, nX = np.linalg.norm(PA[live], axis=1), np.linalg.norm(PX[live], axis=1)
    capA, capX = capacity(PA)[live], capacity(PX)[live]
    R, _ = orthogonal_procrustes(PA[live], PX[live])           # global rotation: PA @ R ~ PX
    PAR = PA[live] @ R
    out = dict(log_norm_ratio=np.log(np.maximum(nX, 1e-30) / np.maximum(nA, 1e-30)),
               cap_A=capA, cap_X=capX, log_cap_ratio=np.log(np.maximum(capX, 1e-30) / np.maximum(capA, 1e-30)),
               rot_total=angle(PA[live], PX[live]), rot_global=angle(PA[live], PAR), rot_local=angle(PAR, PX[live]))
    if W is not None:
        gA, gX = PA[live] @ W.T, PX[live] @ W.T
        out["misalign"] = np.abs(gX - gA).sum(1) / np.maximum(np.abs(gA).sum(1), 1e-30)
    return out

# ---------- one full transition (A -> X), using everything above ----------
def measure_transition(F, live, important, nbrs, alpha, XA, XX, W=None):
    """F: codes from model A (dict train/probe/test/held). XA, XX: activations dicts of model A and model X."""
    Phi_A, _ = anchored_vectors(F["held"], XA["held"])
    Phi_X, _ = anchored_vectors(F["held"], XX["held"])
    rd = readability(XA["probe"], XA["test"], XX["probe"], XX["test"], F["probe"][:, live], F["test"][:, live], alpha)
    pA = partial_r2(XA["probe"], XA["test"], F["probe"][:, live], F["test"][:, live], nbrs, alpha)
    pX = pA if XX is XA else partial_r2(XX["probe"], XX["test"], F["probe"][:, live], F["test"][:, live], nbrs, alpha)
    geo = geometry(Phi_A, Phi_X, np.sqrt((F["held"] ** 2).mean(0)), live, W)
    df = pd.DataFrame(dict(feature=live, important=np.isin(live, important), **rd, U_own=pA - pX, **geo))
    return df, Phi_A, Phi_X

### Demo on model seed 0, SAE seed 0

We first run the measurement once, for the real transition A → B, so you can see what the output looks like.

In [ ]:
def prepare_run(ms, ss):
    """Everything that depends only on model A and the SAE: codes, live/important features, neighbours, probe strength."""
    acts = ACTS[ms]
    sae, scale, l1 = get_sae(ms, ss, acts["A"]["A_train"])
    W, b = HEAD_A[ms]
    F = dict(probe=codes(sae, acts["A"]["A_probe"], scale), test=codes(sae, acts["A"]["A_test"], scale),
             held=codes(sae, acts["A"]["held"], scale))
    Phi_A, _ = anchored_vectors(F["held"], acts["A"]["held"])
    live, top, imp = select_features(F["test"], acts["A"]["A_test"], acts["A"]["y_test"], Phi_A, W, b)
    nbrs = neighbours(F["probe"][:, live], CFG["N_NEIGHBOURS"])
    alpha = choose_alpha(acts["A"]["A_probe"], F["probe"][:, live])
    X = {ck: dict(probe=acts[ck]["A_probe"], test=acts[ck]["A_test"], held=acts[ck]["held"]) for ck in acts}
    dec = sae.dec.weight.detach().numpy().T.astype(np.float64)       # (latents, d): decoder directions
    return dict(sae=sae, F=F, live=live, top=top, imp=imp, nbrs=nbrs, alpha=alpha, X=X, W=W, dec=dec)

t0 = time.time()
P0 = prepare_run(0, 0)
demo, _, _ = measure_transition(P0["F"], P0["live"], P0["top"], P0["nbrs"], P0["alpha"], P0["X"]["A"], P0["X"]["B"], P0["W"])
print(f"live features: {len(P0['live'])} | analysed (top {CFG['TOP_FRAC']:.0%}): {len(P0['top'])} | "
      f"probe alpha: {P0['alpha']:.3g} | {time.time() - t0:.0f}s")

# [TRACKIO] ADDED: log the demo's headline numbers (live / important feature counts, probe strength)
# and its top-10 important features ranked by U as a table.
tlog({"demo_ms0_ss0/n_live": len(P0["live"]), "demo_ms0_ss0/n_important": len(P0["top"]),
      "demo_ms0_ss0/probe_alpha": P0["alpha"],
      "tables/demo_top10_by_U": ttable(demo[demo.important].sort_values("U", ascending=False).head(10), nd=3)})
demo[demo.important].sort_values("U", ascending=False).head(10).round(3)

## Step 9 — Null controls and thresholds

* **A vs A** (pipeline unit test): compare model A with itself. U, M, U_own, the log-norm ratio and the log-capacity ratio must be **exactly zero**.
* **A vs A′** (noise floor): fate thresholds are the 95th percentile of each statistic on this null. Nothing is hand-picked.
* **Noise gate**: the median U of important features on the null must be at most 25% of the median U on the real A → B transition.

In [ ]:
def thresholds_from_null(null_df):
    if len(null_df) == 0:
        raise ValueError("No live features: the SAE has too few active latents. Lower MIN_ACTIVATIONS or retrain the SAE.")
    q = CFG["NULL_PCT"]
    return dict(U=np.percentile(null_df.U, q), M=np.percentile(null_df.M, q),
                norm=np.percentile(null_df.log_norm_ratio, q),
                sd_norm=max(null_df.log_norm_ratio.std(), 1e-12),
                sd_cap=max(null_df.log_cap_ratio.std(), 1e-12))

def zero_test(df):
    cols = ["U", "M", "U_own", "log_norm_ratio", "log_cap_ratio"]
    return float(np.nanmax(np.abs(df[cols].values)))

def noise_gate(null_df, real_df):
    mn, mr = null_df[null_df.important].U.median(), real_df[real_df.important].U.median()
    return dict(median_U_null=mn, median_U_real=mr, passed=bool(mr > 0 and mn <= CFG["NOISE_GATE"] * mr))

null_AA, _, _ = measure_transition(P0["F"], P0["live"], P0["top"], P0["nbrs"], P0["alpha"], P0["X"]["A"], P0["X"]["A"], P0["W"])
null_AAp, _, _ = measure_transition(P0["F"], P0["live"], P0["top"], P0["nbrs"], P0["alpha"], P0["X"]["A"], P0["X"]["Aprime"], P0["W"])
print(f"A vs A  -> max |metric| = {zero_test(null_AA):.2e}   (must be ~0)")
print("thresholds from A vs A':", {k: round(v, 4) for k, v in thresholds_from_null(null_AAp).items()})
print("noise gate:", noise_gate(null_AAp, demo))

# [TRACKIO] ADDED: log the seed-0 null results: the A-vs-A unit test (must be 0), the thresholds set by the
# A-vs-A' null (95th percentiles and spreads), and the noise gate (null median U vs real median U).
_thr0, _gate0 = thresholds_from_null(null_AAp), noise_gate(null_AAp, demo)
tlog({"null_ms0_ss0/zero_test_max": zero_test(null_AA),
      **{f"null_ms0_ss0/thr_{k}": v for k, v in _thr0.items()},
      "null_ms0_ss0/median_U_null": _gate0["median_U_null"], "null_ms0_ss0/median_U_real": _gate0["median_U_real"],
      "null_ms0_ss0/gate_passed": _gate0["passed"]})

## Step 10 — Fate labels

In priority order: **degraded** (U above threshold; sub-typed *fading* or *overlap*, whichever drop is larger in null-standardised units), **misaligned** (M above threshold), **amplified** (norm growth above threshold), otherwise **retained**.

In [ ]:
def fate_labels(df, thr):
    fade_score = -df.log_norm_ratio / thr["sd_norm"]
    ovl_score = -df.log_cap_ratio / thr["sd_cap"]
    lab = np.where(df.U > thr["U"], np.where(fade_score >= ovl_score, "degraded:fading", "degraded:overlap"),
          np.where(df.M > thr["M"], "misaligned",
          np.where(df.log_norm_ratio > thr["norm"], "amplified", "retained")))
    return lab

thr0 = thresholds_from_null(null_AAp)
demo["fate"] = fate_labels(demo, thr0)
null_AAp["fate"] = fate_labels(null_AAp, thr0)
print("Real transition A -> B (important features):")
print(demo[demo.important].fate.value_counts().to_string())
print("\nNull A -> A' (important features) — should be mostly 'retained':")
print(null_AAp[null_AAp.important].fate.value_counts().to_string())

# [TRACKIO] ADDED: log the seed-0 fate counts (how many important features got each label), real vs null.
tlog({**{f"fate_count_ms0_ss0_AB/{k}": v for k, v in demo[demo.important].fate.value_counts().items()},
      **{f"fate_count_ms0_ss0_null/{k}": v for k, v in null_AAp[null_AAp.important].fate.value_counts().items()}})

### Run Steps 5–10 for every model seed × SAE seed

This is the main loop. For each pair it measures the three transitions (A→A, A→A′, A→B), sets thresholds from A→A′, and labels the features. All results go into one table, `RES`, saved to `OUT_DIR/all_features.csv`.

In [ ]:
RUNS, rows, checks = {}, [], []
t0 = time.time()
for ms in range(CFG["N_MODEL_SEEDS"]):
    for ss in range(CFG["N_SAE_SEEDS"]):
        P = P0 if (ms, ss) == (0, 0) else prepare_run(ms, ss)
        dfs = {}
        for name in ["A", "Aprime", "B"]:
            dfs[name], _, _ = measure_transition(P["F"], P["live"], P["top"], P["nbrs"], P["alpha"],
                                                 P["X"]["A"], P["X"][name], P["W"])
        thr = thresholds_from_null(dfs["Aprime"])
        for name, d in dfs.items():
            d["fate"] = fate_labels(d, thr)
            d["model_seed"], d["sae_seed"], d["transition"] = ms, ss, {"A": "A_vs_A", "Aprime": "A_vs_Aprime", "B": "A_vs_B"}[name]
            rows.append(d)
        gate = noise_gate(dfs["Aprime"], dfs["B"])
        checks.append(dict(model_seed=ms, sae_seed=ss, n_live=len(P["live"]), n_analysed=len(P["top"]),
                           zero_test_max=zero_test(dfs["A"]), **gate))
        RUNS[(ms, ss)] = dict(dec=P["dec"], F_held=P["F"]["held"][::2].astype(np.float32), live=P["live"], top=P["top"])
        if ss == 0 or ss == CFG["N_SAE_SEEDS"] - 1:
            print(f"  model seed {ms}, SAE seed {ss} done ({time.time() - t0:.0f}s)")
        if (ms, ss) != (0, 0):
            del P

RES = pd.concat(rows, ignore_index=True)
CHECKS = pd.DataFrame(checks)
RES.to_csv(f"{OUT}/all_features.csv", index=False)
CHECKS.to_csv(f"{OUT}/run_checks.csv", index=False)

# [TRACKIO] ADDED: per-seed checks. step = SAE seed, so each metric draws one line across SAE seeds 0..N-1
# (the "seeds_ms<n>" prefix keeps model seeds separate). Also logs the full run_checks table.
for _, r in CHECKS.iterrows():
    _p = f"seeds_ms{int(r.model_seed)}"
    tlog({f"{_p}/n_live": r.n_live, f"{_p}/n_analysed": r.n_analysed, f"{_p}/zero_test_max": r.zero_test_max,
          f"{_p}/median_U_null": r.median_U_null, f"{_p}/median_U_real": r.median_U_real,
          f"{_p}/gate_passed": r.passed}, step=int(r.sae_seed))
tlog({"tables/run_checks": ttable(CHECKS)})
CHECKS.round(4)

In [ ]:
imp_B = RES[(RES.transition == "A_vs_B") & RES.important]
imp_null = RES[(RES.transition == "A_vs_Aprime") & RES.important]
fate_table = pd.concat([imp_B.fate.value_counts(normalize=True).rename("A -> B"),
                        imp_null.fate.value_counts(normalize=True).rename("A -> A' (null)")], axis=1).fillna(0)
print("Share of important features per fate label, pooled over all seeds:")

# [TRACKIO] ADDED: pooled share of each fate label (real A->B vs null A->A'), as a table and as numbers.
tlog({"tables/fate_shares": ttable(fate_table, nd=3, keep_index=True),
      **{f"fate_share_AB/{k}": v for k, v in fate_table["A -> B"].items()},
      **{f"fate_share_null/{k}": v for k, v in fate_table["A -> A' (null)"].items()}})
fate_table.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, col, lab in zip(ax, ["U", "M", "log_norm_ratio"], ["U (information lost)", "M (information moved)", "log-norm ratio"]):
    a.hist(imp_null[col], bins=30, alpha=0.6, label="A -> A' (null)", density=True)
    a.hist(imp_B[col], bins=30, alpha=0.6, label="A -> B (real)", density=True)
    a.set_title(lab); a.legend()
plt.tight_layout()
# [TRACKIO] ADDED: send this exact figure to the dashboard BEFORE plt.show() closes it, plus native Trackio
# histograms of the same three quantities (real vs null) so different runs can be compared side by side.
tlog({"figures/hist_U_M_lognorm": tfig(fig),
      **{f"hist_real/{c}": thist(imp_B[c].values) for c in ["U", "M", "log_norm_ratio"]},
      **{f"hist_null/{c}": thist(imp_null[c].values) for c in ["U", "M", "log_norm_ratio"]}})
plt.show()

## Step 11 — Reliability across SAE seeds

For each model seed, every SAE seed's latents are matched one-to-one to SAE seed 0's latents (Hungarian algorithm on 0.5 × decoder cosine + 0.5 × activation correlation; a match counts only if decoder cosine ≥ 0.7). Then, for the important features of seed 0:

* **match rate** — the fraction that have a match in another seed;
* **ICC(2,1)** — whether each metric gives the same value across seeds (1 = perfectly consistent), computed on features matched in *all* seeds and always reported next to the match rate;
* **Fleiss' κ** — agreement on fate labels;
* **variance components** — how much run-level results vary because of the model seed versus the SAE seed.

In [ ]:
METRICS = ["U", "M", "U_own", "log_norm_ratio", "log_cap_ratio", "rot_local", "misalign"]
FUNCTIONAL, GEOMETRIC = ["U", "M", "U_own"], ["log_norm_ratio", "log_cap_ratio", "rot_local", "misalign"]

def match_latents(dec0, dec1, F0, F1):
    a = dec0 / np.maximum(np.linalg.norm(dec0, axis=1, keepdims=True), 1e-12)
    b = dec1 / np.maximum(np.linalg.norm(dec1, axis=1, keepdims=True), 1e-12)
    cos = a @ b.T
    z0 = (F0 - F0.mean(0)) / np.maximum(F0.std(0), 1e-12)
    z1 = (F1 - F1.mean(0)) / np.maximum(F1.std(0), 1e-12)
    corr = z0.T @ z1 / len(F0)
    r, c = linear_sum_assignment(-(0.5 * cos + 0.5 * corr))
    mapping = np.full(len(dec0), -1)
    ok = cos[r, c] >= CFG["RULE_MATCH_COS"]
    mapping[r[ok]] = c[ok]
    return mapping

def icc21(Y):
    """ICC(2,1): two-way random effects, absolute agreement, single measurement. Y: (features, seeds)."""
    n, k = Y.shape
    if n < 3:
        return np.nan
    g = Y.mean()
    ssr = k * ((Y.mean(1) - g) ** 2).sum(); ssc = n * ((Y.mean(0) - g) ** 2).sum()
    sse = ((Y - g) ** 2).sum() - ssr - ssc
    msr, msc, mse = ssr / (n - 1), ssc / (k - 1), sse / ((n - 1) * (k - 1))
    den = msr + (k - 1) * mse + k * (msc - mse) / n
    return float((msr - mse) / den) if den > 0 else np.nan

def fleiss_kappa(labels):
    """labels: (items, raters) array of category strings."""
    if len(labels) < 3:
        return np.nan
    cats = np.unique(labels); k = labels.shape[1]
    counts = np.stack([(labels == c).sum(1) for c in cats], 1)
    P_i = ((counts ** 2).sum(1) - k) / (k * (k - 1))
    p_j = counts.sum(0) / counts.sum()
    Pe = (p_j ** 2).sum()
    return float((P_i.mean() - Pe) / (1 - Pe)) if Pe < 1 else np.nan

LOOK = RES[RES.transition == "A_vs_B"].set_index(["model_seed", "sae_seed", "feature"])
match_rates, icc_rows, label_rows, icc_seed = [], {m: [] for m in METRICS}, [], []
icc_live_rows = {m: [] for m in METRICS}     # diagnostic: same, over ALL matched live features
for ms in range(CFG["N_MODEL_SEEDS"]):
    R0 = RUNS[(ms, 0)]
    maps = {ss: match_latents(R0["dec"], RUNS[(ms, ss)]["dec"], R0["F_held"], RUNS[(ms, ss)]["F_held"])
            for ss in range(1, CFG["N_SAE_SEEDS"])}
    for ss, mp in maps.items():
        match_rates.append(dict(model_seed=ms, sae_seed=ss,
                                match_rate_important=float(np.mean(mp[R0["top"]] >= 0)),
                                match_rate_live=float(np.mean(mp[R0["live"]] >= 0))))
    for i in R0["live"]:
        keys = [(ms, 0, i)] + [(ms, ss, maps[ss][i]) for ss in maps]
        if all(kk[2] >= 0 and kk in LOOK.index for kk in keys):
            rows_i = LOOK.loc[keys]
            target = icc_rows if i in set(R0["top"]) else None
            for m in METRICS:
                icc_live_rows[m].append(rows_i[m].values)
                if target is not None:
                    icc_rows[m].append(rows_i[m].values)
            if target is not None:
                label_rows.append(rows_i["fate"].values); icc_seed.append(ms)

MATCH = pd.DataFrame(match_rates)
match_important = MATCH.match_rate_important.mean()
n_matched_all = len(label_rows)
ICC = {m: icc21(np.array(v)) if len(v) else np.nan for m, v in icc_rows.items()}
KAPPA = fleiss_kappa(np.array(label_rows)) if label_rows else np.nan
print(f"Mean match rate, important features: {match_important:.2f} | live features: {MATCH.match_rate_live.mean():.2f}")
print(f"Important features matched in ALL SAE seeds (rows used for ICC/kappa): {n_matched_all}")
print(f"Fleiss' kappa on fate labels: {KAPPA:.3f}")

# [TRACKIO] ADDED: seed-stability results: match rates, number of features matched in ALL SAE seeds, Fleiss' kappa
# (fate-label agreement across SAE seeds), the ICC of every metric (1 = same value in every SAE seed), and the
# per-seed match-rate table.
tlog({"reliability/match_rate_important": match_important, "reliability/match_rate_live": MATCH.match_rate_live.mean(),
      "reliability/n_matched_all_seeds": n_matched_all, "reliability/fleiss_kappa": KAPPA,
      **{f"icc/{m}": v for m, v in ICC.items()}, "tables/match_rates": ttable(MATCH)})
pd.Series(ICC, name="ICC(2,1) across SAE seeds").round(3)

In [ ]:
# Variance components: how much of the run-to-run variation comes from the model seed vs the SAE seed
def run_summary(d):
    deg = d.fate.str.startswith("degraded")
    return pd.Series(dict(median_U=d.U.median(), median_M=d.M.median(), median_log_norm=d.log_norm_ratio.median(),
                          frac_degraded=deg.mean(),
                          frac_fading_of_degraded=(d.fate == "degraded:fading").sum() / max(deg.sum(), 1)))

SUMM = pd.DataFrame([dict(model_seed=k[0], sae_seed=k[1], **run_summary(d))
                     for k, d in imp_B.groupby(["model_seed", "sae_seed"])])
vc = []
for col in ["median_U", "median_M", "median_log_norm", "frac_degraded", "frac_fading_of_degraded"]:
    grp = SUMM.groupby("model_seed")[col]
    var_sae = grp.var(ddof=1).mean()
    var_model = max(grp.mean().var(ddof=1) - var_sae / CFG["N_SAE_SEEDS"], 0.0) if CFG["N_MODEL_SEEDS"] > 1 else np.nan
    tot = var_sae + (var_model if not np.isnan(var_model) else 0)
    vc.append(dict(summary=col, mean=SUMM[col].mean(), var_model_seed=var_model, var_sae_seed=var_sae,
                   share_sae_seed=var_sae / tot if tot > 0 else np.nan))
VARCOMP = pd.DataFrame(vc)

# [TRACKIO] ADDED: variance-components table. With 1 model seed the model-seed column is "n/a" by design.
tlog({"tables/variance_components": ttable(VARCOMP)})
VARCOMP.round(4)

## Step 12 — Validation on a toy world with known features

We build a linear "network" whose activations are exactly *codes × true feature vectors + fixed per-input noise*. Because we chose the true vectors, we know the truth. The "B" model changes them in controlled ways:

| Group | What we do to the true vector |
|---|---|
| fading | shrink its length to 20–70% |
| overlap | bend it toward another feature (length unchanged) |
| rotation | rotate it 30–75° (length unchanged) |
| unchanged | nothing beyond the small background drift every feature gets (also in A′) |

The **true** metrics and **true** labels are computed by applying the same functions to the true codes and true vectors (with thresholds from the true A→A′ null). The measured results, which only see the SAE, are then scored against them: Spearman correlation per metric, label accuracy, and wrong-subtype rates. Note that a feature in the "unchanged" group can still truly be affected, e.g. when another feature is bent toward it.

Three scenarios are run: **mixed** (all groups), **overlap_only** and **fading_only**. The exact same SAE + measurement code as for the CNN is used.

In [ ]:
def make_toy(seed, scenario, d=32, m=64, p=0.06, noise=0.1, jitter=0.05, n_probe=10000, n_test=8000, n_held=6000, k=12):
    g = np.random.RandomState(seed)
    Phi = g.randn(m, d); Phi /= np.linalg.norm(Phi, axis=1, keepdims=True)
    sample = lambda n: (g.rand(n, m) < p) * g.uniform(0.5, 1.5, (n, m))
    Ft = dict(probe=sample(n_probe), test=sample(n_test), held=sample(n_held))
    N = {s: noise * g.randn(len(Ft[s]), d) for s in Ft}            # fixed per-input nuisance
    groups = np.array(["unchanged"] * m, dtype=object)
    perm = g.permutation(m)
    todo = {"mixed": ["fading", "overlap", "rotation"], "overlap_only": ["overlap"], "fading_only": ["fading"]}[scenario]
    for gi, name in enumerate(todo):
        groups[perm[gi * k:(gi + 1) * k]] = name
    # background drift: every feature moves a little, in A' (the null) and in B alike
    PhiAp = Phi + jitter * g.randn(m, d) / np.sqrt(d)
    PhiB = Phi + jitter * g.randn(m, d) / np.sqrt(d)
    free = list(np.where(groups == "unchanged")[0]); g.shuffle(free)
    for i in np.where(groups == "fading")[0]:                       # graded strengths, so the truth is not just 0/1
        PhiB[i] *= g.uniform(0.2, 0.7)
    for i in np.where(groups == "overlap")[0]:
        j = free.pop(); groups[j] = "overlap_partner"
        v = PhiB[i] + g.uniform(0.4, 1.5) * Phi[j]; PhiB[i] = v / np.linalg.norm(v) * np.linalg.norm(PhiB[i])
    for i in np.where(groups == "rotation")[0]:
        r = g.randn(d); r -= (r @ Phi[i]) * Phi[i]; r /= np.linalg.norm(r)
        th = np.radians(g.uniform(30, 75))
        PhiB[i] = (np.cos(th) * Phi[i] + np.sin(th) * r) * np.linalg.norm(PhiB[i])
    acts = lambda P: {s: Ft[s] @ P + N[s] for s in Ft}
    return dict(Ft=Ft, Phi=Phi, PhiAp=PhiAp, PhiB=PhiB, groups=groups, X=dict(A=acts(Phi), Aprime=acts(PhiAp), B=acts(PhiB)))

def toy_truth(T, which):
    """True metrics: the same functions, applied to the TRUE codes and TRUE feature vectors."""
    XA, XX = T["X"]["A"], T["X"][which]
    Ft = T["Ft"]; all_f = np.arange(T["Phi"].shape[0])
    nb_t = neighbours(Ft["probe"], CFG["N_NEIGHBOURS"])
    alpha_t = choose_alpha(XA["probe"], Ft["probe"])
    rd = readability(XA["probe"], XA["test"], XX["probe"], XX["test"], Ft["probe"], Ft["test"], alpha_t)
    own = partial_r2(XA["probe"], XA["test"], Ft["probe"], Ft["test"], nb_t, alpha_t) - \
          partial_r2(XX["probe"], XX["test"], Ft["probe"], Ft["test"], nb_t, alpha_t)
    PX = T["PhiAp"] if which == "Aprime" else T["PhiB"]
    geo = geometry(T["Phi"], PX, np.sqrt((Ft["held"] ** 2).mean(0)), all_f)
    return pd.DataFrame(dict(true_feature=all_f, group=T["groups"], U=rd["U"], M=rd["M"], U_own=own, **geo))

def run_toy(seed, scenario):
    T = make_toy(seed, scenario)
    XA = T["X"]["A"]
    sae, scale, _ = get_sae(seed, 0, XA["probe"], target_l0=4, tag=f"toy_{scenario}", epochs=40, min_steps=8000)
    F = {s: codes(sae, XA[s], scale) for s in ["probe", "test", "held"]}
    live = np.where((F["test"] > 0).sum(0) >= CFG["MIN_ACTIVATIONS"])[0]
    nbrs = neighbours(F["probe"][:, live], CFG["N_NEIGHBOURS"])
    alpha = choose_alpha(XA["probe"], F["probe"][:, live])
    meas = {}
    for name in ["A", "Aprime", "B"]:
        meas[name], _, _ = measure_transition(F, live, live, nbrs, alpha, XA, T["X"][name])
    meas["B"]["fate"] = fate_labels(meas["B"], thresholds_from_null(meas["Aprime"]))
    truth = toy_truth(T, "B")
    truth["true_fate"] = fate_labels(truth, thresholds_from_null(toy_truth(T, "Aprime")))
    # match SAE latents to true features by decoder direction
    dec = sae.dec.weight.detach().numpy().T[live]
    cos = (dec / np.linalg.norm(dec, axis=1, keepdims=True)) @ T["Phi"].T
    r, c = linear_sum_assignment(-cos)
    ok = cos[r, c] >= CFG["RULE_MATCH_COS"]
    mm = meas["B"].iloc[r[ok]].reset_index(drop=True)
    tt = truth.iloc[c[ok]].reset_index(drop=True)
    out = dict(seed=seed, scenario=scenario, n_true=len(truth), n_recovered=int(ok.sum()),
               zero_test=zero_test(meas["A"]), label_accuracy=float((mm.fate == tt.true_fate).mean()))
    for col in ["U", "M", "U_own", "log_norm_ratio", "log_cap_ratio", "rot_local", "rot_total"]:
        out[f"spearman_{col}"] = spearmanr(mm[col], tt[col]).correlation
    lab = pd.DataFrame(dict(group=tt.group, true_fate=tt.true_fate, fate=mm.fate, seed=seed, scenario=scenario))
    return out, lab

t0 = time.time()
toy_out, toy_lab = [], []
for scenario in ["mixed", "overlap_only", "fading_only"]:
    for s in range(CFG["TOY_SEEDS"]):
        print(scenario, s)
        o, l = run_toy(500 + s, scenario); toy_out.append(o); toy_lab.append(l)
TOY = pd.DataFrame(toy_out)
TOYLAB = pd.concat(toy_lab, ignore_index=True)
print(f"toy world done in {time.time() - t0:.0f}s")

# [TRACKIO] ADDED: toy-world validation. Logs the full per-seed / per-scenario table, the MIXED-scenario averages
# (the numbers the reliability rule uses) and how long the toy world took.
_toy_mix = TOY[TOY.scenario == "mixed"].mean(numeric_only=True)
tlog({"tables/toy_world": ttable(TOY, nd=3), "toy/runtime_s": time.time() - t0,
      **{f"toy_mixed/{c}": v for c, v in _toy_mix.items()
         if c.startswith("spearman") or c in ("label_accuracy", "n_recovered")}})
TOY.round(3)

In [ ]:
print("All scenarios: measured label (columns) versus TRUE label (rows); rows sum to 1, a perfect method is diagonal")
TOY_CONF = pd.crosstab(TOYLAB.true_fate, TOYLAB.fate, normalize="index").round(2)   # [TRACKIO] CHANGED: kept in a variable so it can be logged
# [TRACKIO] ADDED: log the toy confusion table (rows = true label, columns = measured label).
tlog({"tables/toy_confusion": ttable(TOY_CONF, nd=2, keep_index=True)})
TOY_CONF

In [ ]:
def frac(df, cond):
    return float(cond(df).mean()) if len(df) else np.nan

true_ovl = TOYLAB[TOYLAB.true_fate == "degraded:overlap"]
true_fad = TOYLAB[TOYLAB.true_fate == "degraded:fading"]
true_ret = TOYLAB[TOYLAB.true_fate == "retained"]
TOY_FP = dict(
    fake_fading_when_truly_overlap=frac(true_ovl, lambda d: d.fate == "degraded:fading"),
    fake_overlap_when_truly_fading=frac(true_fad, lambda d: d.fate == "degraded:overlap"),
    truly_retained_labelled_otherwise=frac(true_ret, lambda d: d.fate != "retained"),
)
print("False-positive rates in the toy world (lower is better):")

# [TRACKIO] ADDED: toy false-positive rates (lower is better). NaN entries (e.g. no truly "retained" features) are skipped.
tlog({f"toy_fp/{k}": v for k, v in TOY_FP.items()})
pd.Series(TOY_FP).round(3)

## Applying the reliability rule and checking that the protocol works

A metric becomes a **primary** label only if: ICC ≥ 0.7 across SAE seeds, match rate ≥ 0.5 among important features, and toy-world Spearman ≥ 0.8 (mixed scenario, averaged over toy seeds). Otherwise it is **exploratory**. If neither the norm nor the capacity metric passes, "degraded" is not sub-typed (it becomes *degraded (unattributed)*), and "amplified" is only kept if the norm metric passes.

In [ ]:
toy_mixed = TOY[TOY.scenario == "mixed"].mean(numeric_only=True)
RULE = []
for m in METRICS:
    sp = toy_mixed.get(f"spearman_{m}", np.nan)
    passed = bool(ICC.get(m, np.nan) >= CFG["RULE_ICC"] and match_important >= CFG["RULE_MATCH"]
                  and sp >= CFG["RULE_TOY_SPEARMAN"])
    RULE.append(dict(metric=m, kind="functional" if m in FUNCTIONAL else "geometric",
                     ICC=ICC.get(m, np.nan), match_rate_important=match_important, toy_spearman=sp,
                     status="PRIMARY" if passed else "exploratory"))
RULE = pd.DataFrame(RULE)
RULE.to_csv(f"{OUT}/reliability_rule.csv", index=False)

# [TRACKIO] ADDED: the reliability-rule table, plus a 1/0 flag per metric (1 = PRIMARY, 0 = exploratory).
tlog({"tables/reliability_rule": ttable(RULE, nd=3),
      **{f"primary/{r.metric}": r.status == "PRIMARY" for r in RULE.itertuples()}})
RULE.round(3)

### Diagnostic: why does a metric fail the rule?

For each metric this shows which of the three criteria failed, how many features the ICC was computed on, the ICC within each model seed, and the ICC over **all** matched live features (not just the top 5%). Two common patterns:

* **Low ICC only on important features, higher on all live features** → the top 5% is too small or too homogeneous (ICC is low when features barely differ from each other, even if each one is measured consistently).
* **Low ICC everywhere** → the metric genuinely depends on the SAE seed; it should stay exploratory, or the SAE needs more training / a smaller dictionary.

In [ ]:
icc_seed_arr = np.array(icc_seed)
diag = []
for m in METRICS:
    Y = np.array(icc_rows[m]) if icc_rows[m] else np.zeros((0, CFG["N_SAE_SEEDS"]))
    per_seed = {f"ICC_model_seed_{s}": icc21(Y[icc_seed_arr == s]) if len(Y) else np.nan
                for s in range(CFG["N_MODEL_SEEDS"])}
    YL = np.array(icc_live_rows[m]) if icc_live_rows[m] else np.zeros((0, 2))
    between = np.nanvar(Y.mean(1)) if len(Y) else np.nan             # spread across features
    within = np.nanmean(Y.var(1, ddof=1)) if len(Y) else np.nan      # spread across SAE seeds
    r = RULE.set_index("metric").loc[m]
    failed = [name for name, ok in [("ICC", r.ICC >= CFG["RULE_ICC"]),
                                    ("match rate", r.match_rate_important >= CFG["RULE_MATCH"]),
                                    ("toy Spearman", r.toy_spearman >= CFG["RULE_TOY_SPEARMAN"])] if not ok]
    diag.append(dict(metric=m, failed=", ".join(failed) or "-", n_rows_important=len(Y), ICC_important=r.ICC,
                     ICC_all_live=icc21(YL) if len(YL) else np.nan, n_rows_live=len(YL),
                     var_across_features=between, var_across_sae_seeds=within, **per_seed))
DIAG = pd.DataFrame(diag)
DIAG.to_csv(f"{OUT}/reliability_diagnostic.csv", index=False)
print(f"match rate among important features: {match_important:.2f} (rule needs >= {CFG['RULE_MATCH']})")

# [TRACKIO] ADDED: the "why did a metric fail the rule" diagnostic table.
tlog({"tables/reliability_diagnostic": ttable(DIAG, nd=3)})
DIAG.round(3)

In [ ]:
primary = set(RULE[RULE.status == "PRIMARY"].metric)
def final_label(f):
    if f.startswith("degraded"):
        sub = f.split(":")[1]
        key = "log_norm_ratio" if sub == "fading" else "log_cap_ratio"
        if "log_norm_ratio" not in primary and "log_cap_ratio" not in primary:
            return "degraded (unattributed)"
        return f if key in primary else "degraded (unattributed)"
    if f == "amplified" and "log_norm_ratio" not in primary:
        return "retained"
    return f

RES["final_fate"] = RES.fate.map(final_label)
imp_B = RES[(RES.transition == "A_vs_B") & RES.important]
deg = imp_B[imp_B.fate.str.startswith("degraded")]
fading_share = (deg.fate == "degraded:fading").mean() if len(deg) else np.nan
fading_share_by_seed = deg.assign(is_fading=deg.fate.eq("degraded:fading")).groupby(["model_seed", "sae_seed"])["is_fading"].mean()

icc_func = np.nanmean([ICC[m] for m in ["U", "M"]]); icc_geo = np.nanmean([ICC[m] for m in ["log_norm_ratio", "log_cap_ratio"]])
null_imp = RES[(RES.transition == "A_vs_Aprime") & RES.important]

VERDICT = pd.DataFrame([
    dict(check="1. Pipeline unit test (A vs A exactly zero)", value=f"max |metric| = {CHECKS.zero_test_max.max():.1e}",
         passed=bool(CHECKS.zero_test_max.max() < 1e-8)),
    dict(check="2. Real forgetting present (A accuracy drop after B > 5 pts)", value=f"{acc_df.forgetting_on_A.mean():.3f}",
         passed=bool(acc_df.forgetting_on_A.mean() > 0.05)),
    dict(check="3. Null forgets little (A accuracy change after A' < 2 pts)",
         value=f"{(acc_df.acc_A_after_A - acc_df.acc_A_after_Aprime).mean():.3f}",
         passed=bool((acc_df.acc_A_after_A - acc_df.acc_A_after_Aprime).mean() < 0.02)),
    dict(check="4. Noise gate (null median U <= 25% of real) in most runs", value=f"{CHECKS.passed.mean():.0%} of runs",
         passed=bool(CHECKS.passed.mean() >= 0.5)),
    dict(check="5. Null labels mostly 'retained' (>= 80% of important features)",
         value=f"{(null_imp.fate == 'retained').mean():.0%}", passed=bool((null_imp.fate == "retained").mean() >= 0.8)),
    dict(check="6. Toy: functional metrics recover truth (Spearman U and M >= 0.8)",
         value=f"U {toy_mixed['spearman_U']:.2f}, M {toy_mixed['spearman_M']:.2f}",
         passed=bool(toy_mixed["spearman_U"] >= 0.8 and toy_mixed["spearman_M"] >= 0.8)),
    dict(check="7. Toy: no invented fading/overlap (both wrong-subtype rates <= 20%)",
         value=f"{TOY_FP['fake_fading_when_truly_overlap']:.2f} / {TOY_FP['fake_overlap_when_truly_fading']:.2f}",
         passed=bool(np.nan_to_num(TOY_FP["fake_fading_when_truly_overlap"]) <= 0.2
                     and np.nan_to_num(TOY_FP["fake_overlap_when_truly_fading"]) <= 0.2)),
    dict(check="8. At least one functional metric passes the reliability rule", value=", ".join(sorted(primary & set(FUNCTIONAL))) or "none",
         passed=bool(primary & set(FUNCTIONAL))),
])
VERDICT.to_csv(f"{OUT}/verdict.csv", index=False)

print("=" * 90)
print(VERDICT.to_string(index=False))
print("=" * 90)
core_ok = VERDICT.passed.iloc[[0, 3, 5, 7]].all()
print("PROTOCOL STATUS:", "WORKING" if VERDICT.passed.all() else
      ("WORKING, with warnings (see failed checks)" if core_ok else "NOT YET WORKING (see failed checks)"))
print(f"\nPrimary metrics: {sorted(primary) or 'none'}   |   exploratory: {sorted(set(METRICS) - primary)}")
print(f"H3b-style check (functional ICC - geometric ICC >= 0.10): {icc_func - icc_geo:+.3f} "
      f"-> {'supported' if icc_func - icc_geo >= 0.10 else 'not supported'}")
if len(fading_share_by_seed):
    print(f"Share of degraded important features attributed to FADING: {float(fading_share):.2f} "
          f"(range across seeds {float(fading_share_by_seed.min()):.2f}-{float(fading_share_by_seed.max()):.2f})")
else:
    print("No important feature was labelled 'degraded', so the fading share is undefined.")
print("\nFinal fate labels of important features (A -> B), after the reliability rule:")
print(imp_B.final_fate.value_counts(normalize=True).round(3).to_string())
json.dump(dict(config=CFG, verdict=VERDICT.to_dict("records"), rule=RULE.to_dict("records"),
               toy_false_positives=TOY_FP, fading_share=fading_share, kappa=KAPPA,
               match_rate_important=match_important), open(f"{OUT}/summary.json", "w"), indent=1, default=str)
print(f"\nAll results saved in {OUT}/")

# ===================== [TRACKIO] ADDED: final verdict, one-page report, and close the run =====================
# Logs each of the 8 checks as 1 (pass) / 0 (fail), the overall status, the fading share, the functional-minus-
# geometric ICC gap, the final-fate shares and the verdict table. Then writes a Markdown report (dashboard
# Reports page) and calls trackio.finish(), which marks the run complete. To log again, re-run from the
# [TRACKIO] connect cell, which starts a fresh run.
_status = "WORKING" if VERDICT.passed.all() else ("WORKING, with warnings" if core_ok else "NOT YET WORKING")
tlog({**{f"check/{i + 1}": p for i, p in enumerate(VERDICT.passed)},
      "result/all_checks_passed": VERDICT.passed.all(), "result/core_checks_passed": core_ok,
      "result/fading_share": fading_share, "result/icc_functional_minus_geometric": icc_func - icc_geo,
      "result/n_primary_metrics": len(primary), "result/forgetting_on_A": acc_df.forgetting_on_A.mean(),
      **{f"final_fate_share/{k}": v for k, v in imp_B.final_fate.value_counts(normalize=True).items()},
      "tables/verdict": ttable(VERDICT.assign(passed=VERDICT.passed.map({True: "PASS", False: "FAIL"})))})
_report = "\n".join([
    f"# Verdict: {RUN_NAME}",
    f"**Who:** {WHO} | **Group:** {RUN_GROUP} | **Model seeds:** {CFG['N_MODEL_SEEDS']} | **SAE seeds:** {CFG['N_SAE_SEEDS']}",
    "",
    f"**Protocol status:** {_status}",
    "",
    f"**Primary metrics:** {', '.join(sorted(primary)) or 'none'}",
    f"**Exploratory metrics:** {', '.join(sorted(set(METRICS) - primary))}",
    "",
    f"**Forgetting on A:** {acc_df.forgetting_on_A.mean():.3f} | "
    f"**Fading share of degraded important features:** {fading_share:.2f} | "
    f"**Functional - geometric ICC:** {icc_func - icc_geo:+.3f}",
    "",
    "```",
    VERDICT.to_string(index=False),
    "```",
])
tlog({"report/verdict": tmd(_report)})
if USE_TRACKIO:
    try:
        trackio.finish()
        print(f"Trackio run finished: {RUN_NAME}  ->  {TRACKIO_SERVER_URL}")
    except Exception as e:
        print("WARNING: trackio.finish failed (ignored):", repr(e))

### How to read the verdict

| Check | If it fails, the likely cause |
|---|---|
| 1. Pipeline unit test | A bug: something differs between the "before" and "after" computations |
| 2. Real forgetting | Task B too short or too easy; raise `EPOCHS_B` or the learning rate |
| 3. Null forgets little | The A′ continuation is too aggressive; lower its learning rate |
| 4. Noise gate | Noise floor too high: more data for probes / anchored regression, or the SAE is unstable |
| 5. Null labels | Thresholds unstable; usually fixes itself with more live features |
| 6. Toy recovery | The SAE does not recover features: train longer, adjust `TARGET_L0` |
| 7. Toy subtypes | The fading/overlap attribution is biased; report subtypes as exploratory |
| 8. Reliability rule | Results depend on the SAE seed; use more SAE epochs or a smaller dictionary |

Synthetic-data runs (`DATA = "synthetic"`) are only a software test. Use CIFAR-10 for results.

## Step 13 — Scaling up

1. **Mid-depth layer.** Change `MultiHeadCNN.features` (or add a forward hook) to return the output of the second conv block, flattened over spatial positions. For feature importance at mid layers you need to run the rest of the network, so replace the closed-form logit update in `select_features` with a hooked forward pass.
2. **Other measurement tools.** Add a TopK SAE (replace the ReLU in `SAE.encode` by keeping the top-k pre-activations) and crosscoders (Masip-style TopK + small L1, pure L1, BatchTopK with Latent Scaling), then run them on the same transitions and compare their A→A′ false-positive rates and toy Spearman.
3. **Residual SAE.** Fit a small SAE on *a_B − Φ_B f − b* to represent features B learned newly, and include its latents in the capacity denominators.
4. **Exact replication.** Re-run Masip et al.'s ViT on Split CIFAR-10 with their settings (fresh head, Adam 1e-4, no weight decay, 100 epochs) and report, for each tool, the share of degraded important features attributed to fading. This notebook's final printout reports that share for the anchored ReLU-L1 tool.

In [ ]:
#transfer the params from the original code of the orginal(last week) and then expand
#expand 2 tasks to 5